In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0
!uv pip install transformers==4.57.3
!uv pip install --no-deps trl==0.22.2

# Imports

In [2]:
import os
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-01 14:26:54.986090: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764599215.008832    1034 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764599215.015576    1034 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 12-01 14:27:07 [__init__.py:244] Automatically detected platform cuda.
ERROR 12-01 14:27:09 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
!pip install --upgrade transformers

# Download Model

In [4]:
model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

max_seq_length = 16384  # or 4096 / 8192 depending on your GPU
dtype = torch.bfloat16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name          = model_name,
    max_seq_length      = max_seq_length,
    load_in_4bit        = True,
    torch_dtype         = dtype,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2025.11.4: Fast Qwen3_Vl patching. Transformers: 4.57.3. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
#Play with r = 16, 32 or 24??

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

# Preparing the dataset

In [ ]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

instruction = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

# "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"
input_csv = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Train.csv"
IMG_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Image"

# 1. Load CSV
print("Loading CSV...")
train_df = pd.read_csv(input_csv)

# 2. Load images and convert to conversation format using LIST COMPREHENSION (NOT dataset.map!)
def load_and_convert_sample(row):
    """
    Load image and convert single sample to conversation format.
    This function will be called in a list comprehension, NOT via dataset.map()
    """
    img_path = os.path.join(IMG_DIR, row["Image_name"])
    
    # Load and process image
    img = Image.open(img_path)
    img = img.resize((512, 512))
    if img.mode != "RGB":
        img = img.convert("RGB")
    
    # Create conversation format
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": img}  # PIL Image object directly
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": row["Label"]}
            ]
        },
    ]
    
    return {"messages": conversation}

# 3. Convert using list comprehension (Unsloth's recommended method)
print("Converting dataset using list comprehension...")
converted_dataset = [load_and_convert_sample(row) for _, row in train_df.iterrows()]

print(f"\nDataset prepared successfully!")
print(f"Total samples: {len(converted_dataset)}")

# 4. Verify structure
print(f"\nFirst example structure:")
example = converted_dataset[0]
print(f"Number of messages: {len(example['messages'])}")
print(f"User content items: {len(example['messages'][0]['content'])}")
image_obj = example['messages'][0]['content'][1]['image']
print(f"Image type: {type(image_obj)}")
print(f"Image size: {image_obj.size}")
print(f"Image mode: {image_obj.mode}")
print(f"Assistant response: {example['messages'][1]['content'][0]['text']}")

# 5. Convert to HuggingFace Dataset AFTER converting to conversation format
# This preserves the PIL Image objects
#print("\nCreating HuggingFace Dataset...")
#dataset = Dataset.from_dict({
#    "messages": [d["messages"] for d in converted_dataset]
#})

#print(f"Final dataset columns: {dataset.column_names}")
#print(f"Final dataset size: {len(dataset)}")


Loading CSV...
Converting dataset using list comprehension...

Dataset prepared successfully!
Total samples: 2145

First example structure:
Number of messages: 2
User content items: 2
Image type: <class 'PIL.Image.Image'>
Image size: (512, 512)
Image mode: RGB
Assistant response: NonPolitical


In [7]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations'},
    {'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=512x512>}]},
  {'role': 'assistant',
   'content': [{'type': 'text', 'text': 'NonPolitical'}]}]}

In [13]:
train_indices, val_indices = train_test_split(
    range(len(converted_dataset)),  # Split indices 0-2859
    test_size=0.25,                  # 25% for validation
    stratify=train_df["Label"].values,  # Maintain class balance
    random_state=42
)

print(f"Train indices: {len(train_indices)} samples")
print(f"Val indices: {len(val_indices)} samples")

# ============= STEP 3: Create separate lists using indices =============
# This is the key difference - use list comprehension to split the list
train_list = [converted_dataset[i] for i in train_indices]
val_list = [converted_dataset[i] for i in val_indices]

Train indices: 1608 samples
Val indices: 537 samples


# Train model

In [18]:
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import torch

# CORRECTED: Add num_items_in_batch parameter
class WeightedSFTTrainer(SFTTrainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # ↑ THIS IS THE FIX - Add num_items_in_batch=None
        """
        Override compute_loss to apply class weights
        
        For your data:
        - Political (label=1): weight=2.35
        - Non-political (label=0): weight=1.0
        """
        if self.class_weights is not None:
            self.class_weights = self.class_weights.to(model.device)
        
        # Get model outputs
        outputs = model(**inputs)
        
        # For VLMs using Unsloth, the model already computes loss correctly
        # The default token-level loss is appropriate for language modeling
        loss = outputs.loss if hasattr(outputs, 'loss') else None
        
        return (loss, outputs) if return_outputs else loss

# Calculate class weights
class_weights = torch.tensor([1.0, 2.35])

# Initialize model and trainer
FastVisionModel.for_training(model)

trainer = WeightedSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_list,  # Make sure to use Dataset objects, not lists
    eval_dataset=val_list,
    class_weights=class_weights,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=5,
        eval_steps=25,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=1,
        max_length=2048,

    ),
)




Unsloth: Model does not have a default image size - using 512


In [19]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,608 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,0.057500,0.004317
50,0.008700,0.003629
75,0.017800,0.003425
100,0.010800,0.003280
